# Transformations Numériques des Variables ResStock
Entrée : `metadata_clean.parquet` → Sortie : `metadata_features.parquet`

In [5]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

df = pd.read_parquet(DATA_PROCESSED / 'metadata_clean.parquet')
print('Shape original :', df.shape)
df_feat = df.copy()

Shape original : (549971, 391)


## 1. Isolation — R-values : `"R-11"` → `11.0`

In [6]:
def extract_r(val):
    if pd.isna(val) or str(val).strip() in ('None', 'nan'): return np.nan
    s = str(val)
    if 'Uninsulated' in s: return 0.0
    m = re.search(r'R-?(\d+\.?\d*)', s)
    return float(m.group(1)) if m else np.nan

INSUL_COLS = [
    'in.insulation_wall', 'in.insulation_ceiling', 'in.insulation_roof',
    'in.insulation_floor', 'in.insulation_foundation_wall',
    'in.insulation_rim_joist', 'in.insulation_slab',
]
for col in INSUL_COLS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(extract_r)

print('Exemple insulation_wall :', df_feat['in.insulation_wall'].value_counts().head())

Exemple insulation_wall : in.insulation_wall
0.0     229491
11.0    160353
19.0     68431
7.0      56976
15.0     34720
Name: count, dtype: int64


## 2. Surface habitable — midpoint ft² → m² : `"1000-1499"` → `116.1 m²`

In [7]:
def midpoint_m2(val):
    if pd.isna(val): return np.nan
    s = str(val).replace(',', '')
    if '+' in s:
        return float(s.replace('+', '')) * 0.0929
    m = re.match(r'(\d+)-(\d+)', s)
    if m:
        midpoint = (float(m.group(1)) + float(m.group(2))) / 2
        return round(midpoint * 0.0929, 1)
    return np.nan

if 'in.geometry_floor_area' in df_feat.columns:
    df_feat['in.geometry_floor_area'] = df_feat['in.geometry_floor_area'].apply(midpoint_m2)

print('Surface (m2) — stats :')
print(df_feat['in.geometry_floor_area'].describe().round(1))

Surface (m2) — stats :
count    549971.0
mean        153.1
std          82.4
min          23.2
25%          81.2
50%         116.1
75%         209.0
max         371.6
Name: in.geometry_floor_area, dtype: float64


## 3. Consignes de température — °F → °C : `"68F"` → `20.0°C`

In [8]:
def f_to_c(val):
    if pd.isna(val): return np.nan
    m = re.search(r'(\d+\.?\d*)', str(val))
    return round((float(m.group(1)) - 32) * 5 / 9, 1) if m else np.nan

SP_COLS = [
    'in.heating_setpoint', 'in.cooling_setpoint',
    'in.heating_setpoint_offset_magnitude', 'in.cooling_setpoint_offset_magnitude',
]
for col in SP_COLS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(f_to_c)

print('Consigne chauffage (degC) :', df_feat['in.heating_setpoint'].value_counts().head())
print('Consigne clim (degC)      :', df_feat['in.cooling_setpoint'].value_counts().head())

Consigne chauffage (degC) : in.heating_setpoint
21.1    127983
20.0    112473
22.2     85121
12.8     66637
23.9     51949
Name: count, dtype: int64
Consigne clim (degC)      : in.cooling_setpoint
22.2    109045
21.1    104648
23.9    103578
20.0     63600
24.4     43812
Name: count, dtype: int64


## 4. Efficacité HVAC — `"AFUE, 80%"` → `0.80` | `"SEER, 13"` → `13.0`

In [9]:
def extract_efficiency(val):
    if pd.isna(val): return np.nan
    s = str(val)
    # AFUE / percentage (e.g. "80% AFUE", "100% Efficiency")
    m_pct = re.search(r'(\d+\.?\d*)%', s)
    if m_pct: return round(float(m_pct.group(1)) / 100, 3)
    # HSPF — heating metric for heat pumps (e.g. "ASHP, SEER 10, 6.2 HSPF")
    m_hspf = re.search(r'(\d+\.?\d*)\s*HSPF', s)
    if m_hspf: return float(m_hspf.group(1))
    # SEER / EER — number at end of string (e.g. "AC, SEER 13", "Room AC, EER 10.7")
    m_num = re.search(r'[\s,]+(\d+\.?\d*)\s*$', s)
    if m_num: return float(m_num.group(1))
    return np.nan

EFF_COLS = ['in.hvac_heating_efficiency', 'in.hvac_cooling_efficiency']
for col in EFF_COLS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(extract_efficiency)

print('Heating efficiency :')
print(df_feat['in.hvac_heating_efficiency'].value_counts().sort_index())
print('\nCooling efficiency :')
print(df_feat['in.hvac_cooling_efficiency'].value_counts().sort_index())

Heating efficiency :
in.hvac_heating_efficiency
0.600      17485
0.680      15214
0.760      19159
0.800     154028
0.900       2483
0.925      85157
1.000     107073
6.200       4075
7.700      41979
8.200       5317
8.500      41083
14.000        83
Name: count, dtype: int64

Cooling efficiency :
in.hvac_cooling_efficiency
8.0       4925
8.5       2332
9.8      14047
10.0     32991
10.7     52419
12.0     40194
13.0    159099
15.0     73338
Name: count, dtype: int64


## 5. Variables ordinales simples — string → int

In [10]:
NUM_COLS = [
    'in.geometry_stories',
    'in.bedrooms',
    'in.occupants',
    'in.air_leakage_to_outside_ach50',
]
for col in NUM_COLS:
    if col in df_feat.columns:
        df_feat[col] = pd.to_numeric(df_feat[col], errors='coerce')

print('Stories   :', df_feat['in.geometry_stories'].value_counts().to_dict())
print('Occupants :', df_feat['in.occupants'].value_counts().sort_index().to_dict())

Stories   : {1: 292940, 2: 183548, 3: 42412, 4: 10864, 5: 4309, 6: 3801, 21: 3590, 20: 1145, 12: 1025, 10: 898, 8: 893, 7: 831, 9: 725, 13: 671, 15: 628, 35: 611, 14: 583, 11: 497}
Occupants : {0.0: 66637, 1.0: 132575, 2.0: 167006, 3.0: 75609, 4.0: 61637, 5.0: 28865, 6.0: 11193, 7.0: 3943, 8.0: 1482, 9.0: 512}


## 6. Revenu — midpoint : `"10000-14999"` → `12500`

In [11]:
def midpoint_income(val):
    if pd.isna(val): return np.nan
    s = str(val).replace(',', '').replace('$', '').strip()
    if s.startswith('<'):
        return float(s[1:]) / 2
    if s.endswith('+'):
        return float(s[:-1])
    m = re.match(r'(\d+)-(\d+)', s)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    return pd.to_numeric(s, errors='coerce')

if 'in.income' in df_feat.columns:
    df_feat['in.income'] = df_feat['in.income'].apply(midpoint_income)

print('Revenu (USD) — stats :')
print(df_feat['in.income'].describe().round(0))

Revenu (USD) — stats :
count    483334.0
mean      77435.0
std       57541.0
min        5000.0
25%       32500.0
50%       65000.0
75%      110000.0
max      200000.0
Name: in.income, dtype: float64


## 7. Vintage → âge du bâtiment : `"1980s"` → `44` ans

In [12]:
REFERENCE_YEAR = 2024

def vintage_to_age(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s.startswith('<'):
        decade_mid = 1930
    else:
        m = re.match(r'(\d{4})s?', s)
        if not m:
            return np.nan
        decade_mid = int(m.group(1)) + 5
    return REFERENCE_YEAR - decade_mid

if 'in.vintage' in df_feat.columns:
    df_feat['in.vintage'] = df_feat['in.vintage'].apply(vintage_to_age)
    print('Âge du bâtiment (années) — stats :')
    print(df_feat['in.vintage'].describe().round(1))

Âge du bâtiment (années) — stats :
count    549971.0
mean         49.0
std          25.0
min           9.0
25%          29.0
50%          49.0
75%          69.0
max          94.0
Name: in.vintage, dtype: float64


## 8. Variables binaires Yes/No → 0 / 1

In [13]:
bool_cols = [
    c for c in df_feat.columns
    if set(df_feat[c].dropna().astype(str).str.strip().unique()) <= {'Yes', 'No'}
]

for c in bool_cols:
    df_feat[c] = df_feat[c].map({'Yes': 1, 'No': 0}).astype('Int8')

zero_var = [c for c in bool_cols if df_feat[c].nunique(dropna=True) <= 1]

print(f'Colonnes binaires converties ({len(bool_cols)}) :')
for c in bool_cols:
    pct_yes = df_feat[c].mean() * 100
    flag = '  ← variance nulle' if c in zero_var else ''
    print(f'  {c:<55} {pct_yes:5.1f}% Yes{flag}')

Colonnes binaires converties (11) :
  in.aiannh_area                                            1.6% Yes
  in.cooling_setpoint_has_offset                           35.8% Yes
  in.electric_vehicle_outlet_access                        56.5% Yes
  in.electric_vehicle_ownership                             1.1% Yes
  in.has_pv                                                 1.0% Yes
  in.heating_setpoint_has_offset                           43.8% Yes
  in.hvac_has_ducts                                        77.0% Yes
  in.hvac_has_zonal_electric_heating                        6.4% Yes
  in.hvac_system_is_faulted                                 0.0% Yes  ← variance nulle
  in.hvac_system_is_scaled                                  0.0% Yes  ← variance nulle
  in.water_heater_in_unit                                  85.7% Yes


## 9. Heures de ventilation — `"Hour12"` → `12`

In [14]:
import re

def extract_hour(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r'Hour(\d+)', str(val))
    return int(m.group(1)) if m else np.nan

HOUR_COLS = ['in.bathroom_spot_vent_hour', 'in.range_spot_vent_hour']
for col in HOUR_COLS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(extract_hour)

print('Heures ventilation :', df_feat['in.bathroom_spot_vent_hour'].value_counts().head())

Heures ventilation : in.bathroom_spot_vent_hour
2     33350
6     33350
22    33350
0     33349
1     33349
Name: count, dtype: int64


## 10. Niveaux d'utilisation — `"120%"` → `1.2`

In [15]:
def extract_pct(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r'(\d+)', str(val))
    return int(m.group(1)) / 100 if m else np.nan

USAGE_COLS = [
    'in.clothes_dryer_usage_level',
    'in.clothes_washer_usage_level',
    'in.dishwasher_usage_level',
    'in.cooking_range_usage_level',
]
for col in USAGE_COLS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(extract_pct)

print('Niveaux d\'utilisation :', df_feat['in.clothes_washer_usage_level'].value_counts().head())

Niveaux d'utilisation : in.clothes_washer_usage_level
1.0    274985
0.8    137495
1.2    137491
Name: count, dtype: int64


## 11. Convertir des porcentage 120% ==>1.2 : 

In [16]:

def pourcentage(pr):
    if pd.isna(pr):
        return np.nan
    p=re.search(r'(\d+)', str(pr))
    return int(p.group(1))/100 if p else np.nan

COLS_POURCENTAGE = [ 'in.clothes_dryer_usage_level','in.clothes_washer_usage_level',
                              'in.dishwasher_usage_level','in.cooking_range_usage_level',
                              'in.hot_water_fixtures','in.lighting_interior_use','in.lighting_other_use',
                              'in.refrigerator_usage_level'
                              ]
for col in COLS_POURCENTAGE:
    df_feat[col] = df_feat[col].apply(pourcentage)

print(df_feat[COLS_POURCENTAGE].head())
print(df_feat[COLS_POURCENTAGE].dtypes)

   in.clothes_dryer_usage_level  in.clothes_washer_usage_level  \
0                          0.01                           0.01   
1                          0.01                           0.01   
2                          0.01                           0.01   
3                          0.01                           0.01   
4                          0.00                           0.00   

   in.dishwasher_usage_level  in.cooking_range_usage_level  \
0                       0.01                          0.01   
1                       0.01                          0.01   
2                       0.01                          0.01   
3                       0.01                          0.01   
4                       0.00                          0.00   

   in.hot_water_fixtures  in.lighting_interior_use  in.lighting_other_use  \
0                    1.1                       1.0                    1.0   
1                    1.1                       1.0                    1.0   

## 12. Convesion des colonnes : 'in.cooling_unavailable_days' 
#### (ex: "Never" → 0  ,"1 week" → 7 )


In [17]:
def unavailable_days(days_str):
    if pd.isna(days_str):
        return np.nan
    s = str(days_str).strip().lower()
    if s == 'never':       return 0
    if s == 'year round':  return 365
    m_month = re.search(r'(\d+)\s*month', s)
    if m_month: return int(m_month.group(1)) * 30
    m_week  = re.search(r'(\d+)\s*week',  s)
    if m_week:  return int(m_week.group(1)) * 7
    m_day   = re.search(r'(\d+)\s*day',   s)
    if m_day:   return int(m_day.group(1))
    return np.nan

COLS_UNAVAILABLE_DAYS = ['in.cooling_unavailable_days', 'in.heating_unavailable_days']

for col in COLS_UNAVAILABLE_DAYS:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].apply(unavailable_days).astype('float32')
        # NaN → médiane (cas rares)
        median_val = df_feat[col].median()
        df_feat[col] = df_feat[col].fillna(median_val)
        print(f'{col} → float32 ✓  (médiane={median_val:.0f} jours)')

in.cooling_unavailable_days → float32 ✓  (médiane=0 jours)
in.heating_unavailable_days → float32 ✓  (médiane=0 jours)


## 13. Conversion de 'in.duct_leakage_and_insulation' 
##### on extrait 2 feature le leakage (%) et l-isolation (R-value -> int)

In [18]:

def duct_leakage_pourcentage(val):

    if pd.isna(val) or str(val).strip().lower() == "none":
        return np.nan
    m = re.search(r'(\d+(?:\.\d+)?)\s*%', str(val))

    return float(m.group(1)) / 100 if m else np.nan


def duct_insulation(val):
    if pd.isna(val) or str(val).strip().lower() == "none":
        return 0
    s = str(val).lower()

    if "r-4" in s:
        return 4

    if "r-6" in s:
        return 6

    if "r-8" in s:
        return 8

    if "uninsulated" in s:
        return 0

    return np.nan

COLS_DUCT_LEAKAGE_AND_INSULATION = ['in.duct_leakage_and_insulation']
df_feat['in.duct_leakage'] = df_feat['in.duct_leakage_and_insulation'].apply(duct_leakage_pourcentage)
df_feat['in.duct_insulation'] = df_feat['in.duct_leakage_and_insulation'].apply(duct_insulation)

## 14. Conversion de la localisation des ducts : 'in.duct_location'

In [19]:
def duct_location(val):
    if pd.isna(val):
        return np.nan

    s = str(val).lower()

    if "none" in s:
        return 0

    location = {
        "living": 1,              
        "heated basement": 1,
        "garage": 3,              
        "crawlspace": 4,
        "attic": 5,               
        "unheated basement": 4
    }

    for k, v in location.items():
        if k in s:
            return v

    return np.nan


df_feat['in.duct_location_int'] = df_feat['in.duct_location'].apply(duct_location)


## 15. Conversion de : 'in.eaves' de 2f^t -> m

In [20]:
def eaves(ft_str):
    if pd.isna(ft_str):
        return np.nan

    m = re.search(r'(\d+\.?\d*)', str(ft_str))
    if m:
        feet = float(m.group(1))
        meters = feet * 0.3048
        return meters

    return np.nan

df_feat['in.eaves_length'] = df_feat['in.eaves'].apply(eaves)

## 16. Résumé — colonnes transformées

In [21]:
transformed = (
    INSUL_COLS
    + ['in.geometry_floor_area']
    + SP_COLS
    + EFF_COLS
    + NUM_COLS
    + ['in.income', 'in.vintage']
    + bool_cols
    + HOUR_COLS
    + USAGE_COLS
    + COLS_POURCENTAGE
    + COLS_UNAVAILABLE_DAYS
    + ['in.duct_leakage', 'in.duct_insulation']
    + ['in.duct_location_int']
    + ['in.eaves_length']   
)
transformed = [c for c in transformed if c in df_feat.columns]

print(f'Colonnes transformées : {len(transformed)}\n')
print(f'{"Colonne":<55} {"Type original":>15} {"Type final":>12} {"% NaN final":>12}')
print('-' * 100)
for col in transformed:
    t_orig  = str(df[col].dtype) if col in df.columns else 'new'
    t_new   = str(df_feat[col].dtype)
    pct_nan = df_feat[col].isna().mean() * 100
    print(f'{col:<55} {t_orig:>15} {t_new:>12} {pct_nan:>11.1f}%')

Colonnes transformées : 51

Colonne                                                   Type original   Type final  % NaN final
----------------------------------------------------------------------------------------------------
in.insulation_wall                                                  str      float64         0.0%
in.insulation_ceiling                                               str      float64        57.1%
in.insulation_roof                                                  str      float64         0.0%
in.insulation_floor                                                 str      float64        39.4%
in.insulation_foundation_wall                                       str      float64        48.2%
in.insulation_rim_joist                                             str      float64        48.2%
in.insulation_slab                                                  str      float64        60.6%
in.geometry_floor_area                                              str      float64   

## 17. Sauvegarde → `metadata_features.parquet`

In [22]:
out_path = DATA_PROCESSED / 'metadata_features.parquet'
df_feat.to_parquet(out_path, index=False)
print(f'Sauvegarde : {out_path}')
print(f'Shape final : {df_feat.shape}')

Sauvegarde : \\FS-SOP\Staff-CMA\yzouarhi\Bureau\Data\FlexiMax\data\processed\metadata_features.parquet
Shape final : (549971, 395)
